# Gaming QoE Prediction Analysis

This notebook explores different approaches to predicting Quality of Experience (QoE) in gaming using network traffic features.

## Labeling Strategies
1. **Ping-based QoE**: Primary target - QoE derived from ping latency thresholds
2. **Survey-based QoE**: Self-reported QoE scores from players
    1. tested with five classes
    2. tested with three classes
    3. tested with two classes
3. **Impact-based QoE**: QoE derived from in-game performance impact scores
    1. tested with five classes
    2. tested with three classes
    3. tested with two classes
4. **Performance-based**: Self-reported player performance ratings

In [ ]:
# Import required libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import time
import pickle
import json

# added
from rank import rank_qoe, rank_str_to_int_mapping, rank_impact, rank_qoe_3_classes, rank_str_to_int_mapping_3_classes, rank_impact_3_classes
from charts import plot_confusion_matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from model_utils import make_models, plot_cm

# Set high DPI for better plot quality
plt.rcParams['figure.dpi'] = 400

In [ ]:
# Define network traffic features
features = [
    'src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 
    'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max',
    'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 
    'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max',
    'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 
    'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max',
    'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 
    'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max',
    'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 
    'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max'
]

print(f"Using {len(features)} network traffic features")

In [ ]:
# Load and preprocess the dataset
# subset for code testing
# full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/subset_window_1754898203.csv'

# full dataset
# full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/window_10_1754904064.csv' # Done
# full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/window_1_1756028067.csv' # this one is problematic since we don't have all features of sliding windows
# full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/window_3_1756030271.csv' # Done
# full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/window_5_1756032959.csv' # Done
full_dataset_filepath = '/home/jony/git/Gaming-qoe-research/output/window_15_1756035248.csv' # 

# Create experiment folder once
results_folder = f"results/experiment_{int(time.time())}_{os.path.basename(full_dataset_filepath).replace('.csv', '')}"
os.makedirs(results_folder, exist_ok=True)



dataset = pd.read_csv(full_dataset_filepath)

print(f"Dataset loaded with {dataset.shape[0]} rows and {dataset.shape[1]} columns")

# TODO - understand why there are missing values, until then I will drop them
# Remove rows with missing feature values
dataset.dropna(subset=features, inplace=True)
print(f"Dataset after dropping NaNs: {dataset.shape[0]} rows")

# Display available columns for reference
print("\nAvailable columns:")
print(dataset.columns.tolist())

## Label Generation

We create multiple target variables to compare different QoE prediction approaches:

In [ ]:
# Generate different target labels

# 1. PRIMARY TARGET: QoE based on ping latency thresholds 
    # 1.1 5-classes
# dataset['y_qoe_by_ping'] = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
    # 1.2 3-classes
dataset['y_qoe_by_ping'] = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping_3_classes[rank_qoe_3_classes(ping)])

# 2. QoE based on impact score (inverse relationship - higher impact = better QoE)
    # 1.1 5-classes
# dataset['y_qoe_by_impact'] = dataset['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
    # 1.2 3-classes
# dataset['y_qoe_by_impact'] = dataset['impact'].apply(lambda impact: rank_str_to_int_mapping_3_classes[rank_impact_3_classes(impact)])

# 3. QoE based on self-reported scores
    # 1.1 5-classes
    # The default values in this column already separated to 5 classes
    # 1.2 3-classes
dataset['self_rate_qoe_score'] = dataset['self_rate_qoe_score'].apply(lambda x: 1 if x in [1, 2] else (2 if x == 3 else 3))

# Display label distributions
print("Label distributions:")
print("\nPing-based QoE:")
print(dataset['y_qoe_by_ping'].value_counts().sort_index())
print("\nSurvey-based QoE:")
print(dataset['self_rate_qoe_score'].value_counts().sort_index())
# print("\nImpact-based QoE:")
# print(dataset['y_qoe_by_impact'].value_counts().sort_index())

In [ ]:
# Train/test split strategy: Use every 4th experiment for testing
train_mask = dataset['exp_id'] % 4 != 0
test_mask = dataset['exp_id'] % 4 == 0

X_train = dataset[train_mask][features]
X_test = dataset[test_mask][features]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Total samples: {len(X_train) + len(X_test)}")

## Model Training and Evaluation

We test multiple models on different target variables:

In [ ]:

trained_model = []
result_list = []

# target_list = [
#     ('ping', y_train_by_ping, y_test_by_ping),
#     ('survey', y_train_by_survey, y_test_by_survey)
# ]


# Define target variables to evaluate
target_list = [
    ('Ping-based QoE', 'y_qoe_by_ping'),
    ('Survey-based QoE', 'self_rate_qoe_score')
    # ('Impact-based QoE', 'y_qoe_by_impact'),
    # ('Performance Rating', 'self_rate_player_performance')
]



for target_name, target_col in target_list:
    print(f"\n{'='*50}")
    print(f"TARGET: {target_name}")
    print(f"{'='*50}")
    
    # Prepare target variables from dataset
    y_train_raw = dataset[train_mask][target_col]
    y_test_raw = dataset[test_mask][target_col]

    # Create label variants: original multi-class + binary
    y_train_variants = [
        y_train_raw,  # Original multi-class labels
        y_train_raw.apply(lambda x: 'bad' if x <= 2 else 'good')  # Binary labels
    ]
    y_test_variants = [
        y_test_raw,   # Original multi-class labels  
        y_test_raw.apply(lambda x: 'bad' if x <= 2 else 'good')   # Binary labels
    ]


    for variant_idx, (y_train_current, y_test_current) in enumerate(zip(y_train_variants, y_test_variants)):
        
        # Encode labels to 0..K-1 for sklearn compatibility
        le = LabelEncoder()
        le.fit(pd.concat([y_train_current, y_test_current], axis=0).astype(str))
        y_train_encoded = le.transform(y_train_current.astype(str))
        y_test_encoded = le.transform(y_test_current.astype(str))
        n_classes = len(le.classes_)

        # Build fresh models configured for this number of classes
        list_of_models = make_models(n_classes)

        for model_name, model in list_of_models:
            variant_type = 'multiclass' if variant_idx == 0 else 'binary'
            print(f'Training {model_name} ({variant_type})')
            
            # Clone model to ensure fresh instance
            model = clone(model)
            model.fit(X_train, y_train_encoded)

            print(f'Testing {model_name}')
            y_pred_encoded = model.predict(X_test)
            # Back to original labels for reports and storage
            y_pred_decoded = le.inverse_transform(y_pred_encoded)

            report = classification_report(y_test_current.astype(str), y_pred_decoded.astype(str),
                                           digits=2, output_dict=True)

            trained_model.append({
                'target_name': target_name,
                'model_name': model_name,
                'model': model,
                'label_encoder': le,                    # Keep for later inverse_transform if needed
                'y_test': y_test_current,               # Original labels for this variant
                'y_pred': pd.Series(y_pred_decoded),    # Predictions in original format
                'report': report,
                'variant': variant_type                 # Track which variant (multiclass/binary)
            })

            for k, v in report.items():
                if k not in ('accuracy', 'macro avg', 'weighted avg'):
                    res = {
                        'target': target_name,
                        'model': model_name,
                        'classes': n_classes,
                        'class': k,
                        'precision': round(v['precision'], 2),
                        'recall': round(v['recall'], 2),
                        'f1-score': round(v['f1-score'], 2),
                        'variant': variant_type
                    }
                    result_list.append(res)

result_df = pd.DataFrame(result_list)

### Print resutls
- accuracy
- recall precision f1-score
- confution matrix normalazied and regular

#### print 'accuracy', 'macro avg', 'weighted avg' for each experiment

In [ ]:
# After training is done
for tm in trained_model:
    print(f"\nTarget: {tm['target_name']} - Model: {tm['model_name']}, Variant: {tm['variant']}")
    for k, v in tm['report'].items():
        if k in ('accuracy', 'macro avg', 'weighted avg'):
            print(k, v)


#### accuracy table for all experiment
- sorted
- per experimnet

In [ ]:
# Build accuracy table from the stored reports
rows = []
for tm in trained_model:
    acc = float(tm['report']['accuracy'])
    macro_avg = tm['report']['macro avg']
    weighted_avg = tm['report']['weighted avg']
    
    # Keep classes to distinguish 5-class vs 2-class runs
    n_classes = len(tm['label_encoder'].classes_) if 'label_encoder' in tm and tm['label_encoder'] is not None else tm['y_test'].nunique()
    rows.append({
        'target': tm['target_name'],
        'model': tm['model_name'],
        'classes': n_classes,
        'accuracy': round(acc, 3),
        'macro_precision': round(macro_avg['precision'], 3),
        'macro_recall': round(macro_avg['recall'], 3),
        'macro_f1': round(macro_avg['f1-score'], 3),
        'weighted_precision': round(weighted_avg['precision'], 3),
        'weighted_recall': round(weighted_avg['recall'], 3),
        'weighted_f1': round(weighted_avg['f1-score'], 3),
        'variant': tm['variant']
    })

accuracy_df = pd.DataFrame(rows)

# Option A - global sort by accuracy high to low
print("\n=== Accuracy - all runs, sorted high to low ===")
# print(accuracy_df.sort_values('accuracy', ascending=False).to_string(index=False))
display(accuracy_df.sort_values('accuracy', ascending=False))
accuracy_df.to_csv(os.path.join(results_folder, 'accuracy_summary.csv'), index=False)


# Option B - sorted within each target and class count
print("\n=== Accuracy by target and classes ===")
for (t, c), grp in accuracy_df.groupby(['target', 'classes']):
    print(f"\n{t} - {c} classes")
    print(grp.sort_values('accuracy', ascending=False)[['model', 'accuracy', 'macro_f1', 'weighted_f1']].to_string(index=False))

In [ ]:
# Save best models for each target/variant combination
print("\n=== Saving Best Models ===")

# Find the best model for each target/variant combination
best_models = {}
for tm in trained_model:
    key = (tm['target_name'], tm['variant'])
    accuracy = tm['report']['accuracy']
    
    if key not in best_models or accuracy > best_models[key]['accuracy']:
        best_models[key] = {
            'model_info': tm,
            'accuracy': accuracy
        }

# Save each best model
saved_models_info = []
for (target_name, variant), best_info in best_models.items():
    model_info = best_info['model_info']
    accuracy = best_info['accuracy']
    macro_avg = model_info['report']['macro avg']
    weighted_avg = model_info['report']['weighted avg']
    
    # Create filename safe version of target name
    target_safe = target_name.replace(' ', '_').replace('-', '_')
    model_filename = f"best_model_{target_safe}_{variant}.pkl"
    model_path = os.path.join(results_folder, model_filename)
    
    # Save the model using pickle
    with open(model_path, 'wb') as f:
        pickle.dump({
            'model': model_info['model'],
            'label_encoder': model_info['label_encoder'],
            'features': features,
            'target_name': target_name,
            'model_name': model_info['model_name'],
            'variant': variant,
            'accuracy': accuracy,
            'report': model_info['report']
        }, f)
    
    print(f"Saved: {model_info['model_name']} for {target_name} ({variant}) - Accuracy: {accuracy:.3f}")
    
    # Store info for summary
    saved_models_info.append({
        'target_name': target_name,
        'variant': variant,
        'model_name': model_info['model_name'],
        'accuracy': round(accuracy, 3),
        'macro_precision': round(macro_avg['precision'], 3),
        'macro_recall': round(macro_avg['recall'], 3),
        'macro_f1': round(macro_avg['f1-score'], 3),
        'weighted_precision': round(weighted_avg['precision'], 3),
        'weighted_recall': round(weighted_avg['recall'], 3),
        'weighted_f1': round(weighted_avg['f1-score'], 3),
        'filename': model_filename
    })

# Save a summary of saved models
summary_df = pd.DataFrame(saved_models_info)
summary_path = os.path.join(results_folder, 'best_models_summary.csv')
summary_df.to_csv(summary_path, index=False)

In [ ]:
# First, find the best accuracy for each target/variant combination
best_accuracies = {}
for tm in trained_model:
    key = (tm['target_name'], tm['variant'])
    accuracy = tm['report']['accuracy']
    if key not in best_accuracies or accuracy > best_accuracies[key]:
        best_accuracies[key] = accuracy

for tm in trained_model:
    y_true = tm['y_test'].astype(str)
    y_pred = tm['y_pred'].astype(str)
    
    if 'label_encoder' in tm and tm['label_encoder'] is not None:
        labels_current = list(tm['label_encoder'].classes_)
    else:
        labels_current = sorted(pd.Index(y_true.unique()).astype(str))

    # Check if this is the best model for this target/variant combination
    key = (tm['target_name'], tm['variant'])
    is_best = tm['report']['accuracy'] == best_accuracies[key]
    
    print(f"\nPlotting CMs for Target: {tm['target_name']} - Model: {tm['model_name']} - Variant: {tm['variant']}")
    
    # Plot non-normalized confusion matrix
    save_path_raw = f"{results_folder}/plot_{tm['target_name']}_{tm['model_name']}_{tm['variant']}_raw.png"
    plot_cm(y_true, y_pred, labels=labels_current, normalize=False, title="", 
            show_plot=is_best, save_path=save_path_raw)
    
    # Plot normalized confusion matrix
    save_path_norm = f"{results_folder}/plot_{tm['target_name']}_{tm['model_name']}_{tm['variant']}_normalized.png"
    plot_cm(y_true, y_pred, labels=labels_current, normalize=True, title="", 
            show_plot=is_best, save_path=save_path_norm)

In [ ]:

if not result_df.empty:
    # Create pivot table for better visualization
    pivot_table = result_df.pivot_table(
        index=['target', 'class'], 
        columns='model', 
        values=['precision', 'recall', 'f1-score'],
        aggfunc='first'
    )
    
    print("Results Summary:")
    print(pivot_table)
    
    # Save results to the same folder as plots with unique timestamp
    os.makedirs(results_folder, exist_ok=True)
    pivot_table.to_csv(os.path.join(results_folder, 'qoe_prediction_results.csv'))
    # pivot_table.to_latex('qoe_prediction_results.tex', float_format="%.3f")
    print("\nResults saved to 'qoe_prediction_results.csv' and 'qoe_prediction_results.tex'")
else:
    print("No results to display")

## Confusion Matrix Visualization

Let's visualize the confusion matrices for the best performing models:

In [ ]:
# Plot confusion matrices for key models
if trained_model:
    # Find Random Forest models for ping-based QoE
    ping_rf_models = [
        m for m in trained_model 
        if 'Ping-based' in m['target_name'] and 'Random Forest' in m['model_name']
    ]
    
    if ping_rf_models:
        model_info = ping_rf_models[0]
        y_true = model_info['y_test'].astype(str)
        y_pred = model_info['y_pred'].astype(str)
        
        # Use existing plot_cm function from model_utils
        print(f"\nConfusion Matrix: {model_info['target_name']} - {model_info['model_name']}")
        if 'label_encoder' in model_info and model_info['label_encoder'] is not None:
            labels_current = list(model_info['label_encoder'].classes_)
        else:
            labels_current = sorted(pd.Index(y_true.unique()).astype(str))
        
        plot_cm(y_true, y_pred, labels=labels_current, normalize=False, title="Raw Counts")
        plot_cm(y_true, y_pred, labels=labels_current, normalize=True, title="Normalized")
else:
    print("No trained models available for visualization")

## Feature Importance Analysis

In [ ]:
# Analyze feature importance for Random Forest models
rf_models = [m for m in trained_model if 'Random Forest' in m['model_name']]

for model_info in rf_models:
    if hasattr(model_info['model'], 'feature_importances_'):
        importance_df = pd.DataFrame({
            'feature': features,
            'importance': model_info['model'].feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 10 Features - {model_info['target_name']}:")
        print(importance_df.head(10))
        
        # Plot top features
        plt.figure(figsize=(10, 6))
        top_features = importance_df.head(10)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Feature Importance')
        plt.title(f'Top 10 Feature Importance - {model_info["target_name"]}')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        break  # Show only one for brevity

## Summary and Conclusions

This analysis shows different approaches to QoE prediction:

1. **Ping-based QoE**: Uses network latency thresholds (primary approach)
2. **Survey-based QoE**: Uses player self-reported quality ratings
3. **Impact-based QoE**: Derived from in-game performance metrics
4. **Performance Rating**: Player self-assessment of their performance

The comparison helps understand which network features are most predictive of different aspects of gaming experience.

# Window Size Analysis

This section analyzes how QoE prediction accuracy improves with increasing window sizes (3s, 5s, 10s, 15s).
We compare the best performing models across different window sizes to understand the impact of temporal context.

In [ ]:
# Collect results from all window size experiments
import glob
import re

# Define the experiment folders and their corresponding window sizes
experiment_info = {
    'experiment_1756046719_window_3_1756030271': 3,
    'experiment_1756047026_window_5_1756032959': 5, 
    'experiment_1756112831_window_10_1754904064': 10,
    'experiment_1756047663_window_15_1756035248': 15  # This might be the current running experiment
}

# Collect accuracy data from all completed experiments
window_results = []

for exp_folder, window_size in experiment_info.items():
    summary_path = f"results/{exp_folder}/best_models_summary.csv"
    
    try:
        if os.path.exists(summary_path):
            df = pd.read_csv(summary_path)
            for _, row in df.iterrows():
                if row['variant'] == 'multiclass':  # Focus on multiclass as requested
                    window_results.append({
                        'window_size': window_size,
                        'target_name': row['target_name'],
                        'model_name': row['model_name'],
                        'accuracy': row['accuracy'],
                        'experiment_folder': exp_folder
                    })
            print(f"✓ Loaded results from {window_size}s window experiment")
        else:
            print(f"⚠ No results found for {window_size}s window (experiment may be running)")
    except Exception as e:
        print(f"✗ Error loading {window_size}s window results: {e}")

# Convert to DataFrame for analysis
window_df = pd.DataFrame(window_results)

if not window_df.empty:
    print(f"\nCollected {len(window_df)} results from {window_df['window_size'].nunique()} different window sizes")
    print("\nResults summary:")
    display(window_df.sort_values(['target_name', 'window_size']))
else:
    print("No results collected - experiments may still be running")

In [ ]:
# Check model consistency across window sizes and determine reference model
print("=== Model Consistency Analysis ===")

if not window_df.empty:
    # Check what models are used for each target across window sizes
    for target in window_df['target_name'].unique():
        target_data = window_df[window_df['target_name'] == target]
        print(f"\n{target}:")
        print(target_data[['window_size', 'model_name', 'accuracy']].sort_values('window_size'))
        
        # Check if all models are the same
        unique_models = target_data['model_name'].unique()
        if len(unique_models) == 1:
            print(f"✓ Consistent model across all window sizes: {unique_models[0]}")
        else:
            print(f"⚠ Different models used: {unique_models}")
            
            # Get the best model from 15s window (if available) as reference
            ref_window_data = target_data[target_data['window_size'] == 15]
            if not ref_window_data.empty:
                reference_model = ref_window_data.iloc[0]['model_name']
                print(f"→ Using {reference_model} from 15s window as reference model")
                
                # Filter data to only include this model type across all window sizes
                filtered_data = target_data[target_data['model_name'] == reference_model]
                if len(filtered_data) != len(target_data):
                    print(f"→ Will use {reference_model} results only (available for {len(filtered_data)}/{len(target_data)} window sizes)")
            else:
                print("→ 15s window not available, will use most common model")
                reference_model = target_data['model_name'].value_counts().index[0]
                print(f"→ Using most common model: {reference_model}")

    # Create a filtered dataset for plotting (using consistent models)
    plot_data = []
    
    for target in window_df['target_name'].unique():
        target_data = window_df[window_df['target_name'] == target]
        
        # Determine reference model for this target
        ref_window_data = target_data[target_data['window_size'] == 15]
        if not ref_window_data.empty:
            reference_model = ref_window_data.iloc[0]['model_name']
        else:
            reference_model = target_data['model_name'].value_counts().index[0]
        
        # Keep only data for the reference model
        consistent_data = target_data[target_data['model_name'] == reference_model]
        plot_data.extend(consistent_data.to_dict('records'))
    
    plot_df = pd.DataFrame(plot_data)
    print(f"\n=== Final Dataset for Plotting ===")
    if not plot_df.empty:
        display(plot_df.sort_values(['target_name', 'window_size']))
    else:
        print("No consistent data available for plotting")
else:
    print("No data available for analysis")

In [ ]:
# Create comprehensive summary table
if not window_df.empty:
    print("=== Comprehensive Window Size Comparison ===")
    
    # Create pivot table for better visualization
    pivot_accuracy = window_df.pivot_table(
        index='target_name', 
        columns='window_size', 
        values='accuracy',
        aggfunc='first'
    )
    
    print("\nAccuracy Matrix (rows=targets, columns=window_size):")
    display(pivot_accuracy)
    
    # Create model consistency table
    pivot_models = window_df.pivot_table(
        index='target_name', 
        columns='window_size', 
        values='model_name',
        aggfunc='first'
    )
    
    print("\nModel Type Matrix (rows=targets, columns=window_size):")
    display(pivot_models)
    
    # Save comprehensive results
    if 'results_folder' in locals():
        # Save the collected window analysis data
        window_df.to_csv(os.path.join(results_folder, 'window_size_analysis.csv'), index=False)
        pivot_accuracy.to_csv(os.path.join(results_folder, 'accuracy_by_window_size.csv'))
        pivot_models.to_csv(os.path.join(results_folder, 'models_by_window_size.csv'))
        print(f"\nWindow size analysis saved to {results_folder}/")
    
    # Final summary statistics
    print("\n=== Summary Statistics ===")
    print(f"Total experiments analyzed: {window_df['window_size'].nunique()}")
    print(f"Window sizes tested: {sorted(window_df['window_size'].unique())} seconds")
    print(f"Target types: {list(window_df['target_name'].unique())}")
    print(f"Model types used: {list(window_df['model_name'].unique())}")
    
    if len(window_df) > 0:
        print(f"Accuracy range: {window_df['accuracy'].min():.3f} - {window_df['accuracy'].max():.3f}")
        best_result = window_df.loc[window_df['accuracy'].idxmax()]
        print(f"Best overall result: {best_result['accuracy']:.3f} ({best_result['target_name']}, {best_result['window_size']}s, {best_result['model_name']})")

else:
    print("No window size data available for comprehensive analysis")

# Data Augmentation Analysis

This section explores data augmentation techniques to increase dataset size and improve model performance.
We'll use SMOTE and other synthetic data generation methods to create balanced, larger datasets.

In [ ]:
# Install required libraries for data augmentation
import subprocess
import sys

try:
    from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine import SMOTETomek, SMOTEENN
    print("✓ imbalanced-learn already installed")
except ImportError:
    print("Installing imbalanced-learn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine import SMOTETomek, SMOTEENN
    print("✓ imbalanced-learn installed successfully")

from sklearn.utils import resample
from collections import Counter
import numpy as np
import pandas as pd

print("✓ All augmentation libraries ready")

In [ ]:
# Analyze current class distribution and prepare for augmentation
print("=== Current Class Distribution Analysis ===")

# Analyze class distribution for each target
target_analysis = {}

for target_name, target_col in target_list:
    print(f"\n{target_name} ({target_col}):")
    
    # Get training data for this target
    y_train_current = dataset[train_mask][target_col]
    X_train_current = dataset[train_mask][features]
    
    # Remove any rows with NaN values
    clean_mask = ~(X_train_current.isnull().any(axis=1) | y_train_current.isnull())
    X_clean = X_train_current[clean_mask]
    y_clean = y_train_current[clean_mask]
    
    # Analyze distribution
    class_counts = Counter(y_clean)
    total_samples = len(y_clean)
    
    print(f"  Total training samples: {total_samples}")
    print(f"  Class distribution:")
    for class_val, count in sorted(class_counts.items()):
        percentage = (count / total_samples) * 100
        print(f"    Class {class_val}: {count} samples ({percentage:.1f}%)")
    
    # Calculate imbalance ratio
    max_class = max(class_counts.values())
    min_class = min(class_counts.values())
    imbalance_ratio = max_class / min_class
    print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")
    
    # Store for augmentation
    target_analysis[target_name] = {
        'target_col': target_col,
        'X_train': X_clean,
        'y_train': y_clean,
        'class_counts': class_counts,
        'imbalance_ratio': imbalance_ratio,
        'total_samples': total_samples
    }
    
    # Determine if augmentation is beneficial
    if imbalance_ratio > 2.0:
        print(f"  → High imbalance detected - SMOTE will be beneficial")
    elif total_samples < 5000:
        print(f"  → Small dataset - augmentation can help increase size")
    else:
        print(f"  → Well-balanced dataset - augmentation may still improve generalization")

print(f"\n=== Augmentation Recommendations ===")
for target_name, analysis in target_analysis.items():
    print(f"\n{target_name}:")
    if analysis['imbalance_ratio'] > 3.0:
        print("  → Recommended: SMOTE + BorderlineSMOTE for severe imbalance")
    elif analysis['imbalance_ratio'] > 2.0:
        print("  → Recommended: SMOTE for moderate imbalance")
    else:
        print("  → Recommended: SMOTE for dataset size increase")

In [ ]:
# Apply different augmentation techniques
print("=== Applying Data Augmentation Techniques ===")

# Define augmentation strategies
augmentation_strategies = {
    'SMOTE': SMOTE(random_state=42, k_neighbors=5),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42, k_neighbors=5),
    'ADASYN': ADASYN(random_state=42, n_neighbors=5),
    'SVMSMOTE': SVMSMOTE(random_state=42, k_neighbors=5),
    # Combined strategies for cleaning
    'SMOTE+Tomek': SMOTETomek(random_state=42),
    'SMOTE+EditedNN': SMOTEENN(random_state=42)
}

# Store results for comparison
augmented_datasets = {}

for target_name, analysis in target_analysis.items():
    print(f"\n{'='*60}")
    print(f"AUGMENTING: {target_name}")
    print(f"{'='*60}")
    
    X_original = analysis['X_train']
    y_original = analysis['y_train']
    
    augmented_datasets[target_name] = {
        'original': {
            'X': X_original,
            'y': y_original,
            'size': len(X_original),
            'distribution': Counter(y_original)
        }
    }
    
    print(f"Original dataset: {len(X_original)} samples")
    print(f"Original distribution: {dict(Counter(y_original))}")
    
    # Try each augmentation strategy
    for strategy_name, strategy in augmentation_strategies.items():
        try:
            print(f"\nApplying {strategy_name}...")
            
            # Apply augmentation
            X_augmented, y_augmented = strategy.fit_resample(X_original, y_original)
            
            # Store results
            augmented_datasets[target_name][strategy_name] = {
                'X': X_augmented,
                'y': y_augmented,
                'size': len(X_augmented),
                'distribution': Counter(y_augmented),
                'strategy': strategy
            }
            
            # Calculate increase
            size_increase = len(X_augmented) - len(X_original)
            size_increase_pct = (size_increase / len(X_original)) * 100
            
            print(f"  ✓ Success: {len(X_augmented)} samples (+{size_increase}, +{size_increase_pct:.1f}%)")
            print(f"  New distribution: {dict(Counter(y_augmented))}")
            
        except Exception as e:
            print(f"  ✗ Failed: {str(e)}")
            # Store failure info
            augmented_datasets[target_name][strategy_name] = {
                'error': str(e),
                'size': 0
            }

# Summary of augmentation results
print(f"\n{'='*80}")
print("AUGMENTATION SUMMARY")
print(f"{'='*80}")

for target_name, target_data in augmented_datasets.items():
    print(f"\n{target_name}:")
    original_size = target_data['original']['size']
    
    print(f"  Original: {original_size:,} samples")
    
    # Sort strategies by dataset size increase
    successful_strategies = []
    for strategy_name, strategy_data in target_data.items():
        if strategy_name != 'original' and 'error' not in strategy_data:
            increase = strategy_data['size'] - original_size
            increase_pct = (increase / original_size) * 100
            successful_strategies.append((strategy_name, strategy_data['size'], increase_pct))
    
    successful_strategies.sort(key=lambda x: x[1], reverse=True)
    
    for strategy_name, new_size, increase_pct in successful_strategies[:3]:  # Show top 3
        print(f"  {strategy_name}: {new_size:,} samples (+{increase_pct:.1f}%)")
        
        # Show class balance
        dist = target_data[strategy_name]['distribution']
        balance_info = ", ".join([f"Class {k}: {v}" for k, v in sorted(dist.items())])
        print(f"    Distribution: {balance_info}")

print(f"\n=== Best Augmentation Strategy Recommendations ===")
for target_name, target_data in augmented_datasets.items():
    original_size = target_data['original']['size']
    original_dist = target_data['original']['distribution']
    
    # Find best strategy (good balance + reasonable size increase)
    best_strategy = None
    best_score = 0
    
    for strategy_name, strategy_data in target_data.items():
        if strategy_name != 'original' and 'error' not in strategy_data:
            # Calculate balance score (lower variance is better)
            dist_values = list(strategy_data['distribution'].values())
            balance_score = 1 / (np.std(dist_values) + 1)  # +1 to avoid division by zero
            
            # Calculate size score
            size_score = strategy_data['size'] / original_size
            
            # Combined score (balance and size)
            combined_score = balance_score * min(size_score, 3.0)  # Cap size benefit at 3x
            
            if combined_score > best_score:
                best_score = combined_score
                best_strategy = strategy_name
    
    if best_strategy:
        best_data = target_data[best_strategy]
        increase_pct = ((best_data['size'] - original_size) / original_size) * 100
        print(f"\n{target_name}: Best strategy = {best_strategy}")
        print(f"  Size: {original_size:,} → {best_data['size']:,} (+{increase_pct:.1f}%)")
        print(f"  Balance improvement: {dict(best_data['distribution'])}")
    else:
        print(f"\n{target_name}: No suitable augmentation strategy found")

In [ ]:
# Train models with augmented data and compare performance
print("=== Training Models with Augmented Data ===")

# Select best augmentation strategies for training
selected_strategies = {}
for target_name, target_data in augmented_datasets.items():
    # Choose top 2-3 strategies for this target
    original_size = target_data['original']['size']
    
    strategy_scores = []
    for strategy_name, strategy_data in target_data.items():
        if strategy_name != 'original' and 'error' not in strategy_data:
            # Score based on size increase and balance
            dist_values = list(strategy_data['distribution'].values())
            balance_score = 1 / (np.std(dist_values) + 1)
            size_score = min(strategy_data['size'] / original_size, 3.0)
            combined_score = balance_score * size_score
            strategy_scores.append((strategy_name, combined_score, strategy_data))
    
    # Sort and select top strategies
    strategy_scores.sort(key=lambda x: x[1], reverse=True)
    selected_strategies[target_name] = {
        'original': target_data['original']
    }
    
    # Add top strategies
    for strategy_name, score, strategy_data in strategy_scores[:2]:  # Top 2 strategies
        selected_strategies[target_name][strategy_name] = strategy_data
    
    print(f"\n{target_name} - Selected strategies:")
    for strategy in selected_strategies[target_name].keys():
        if strategy in target_data:
            size = target_data[strategy]['size']
            print(f"  • {strategy}: {size:,} samples")

# Train models with augmented datasets
augmented_results = []
X_test_original = dataset[test_mask][features]

for target_name, strategies in selected_strategies.items():
    target_col = target_analysis[target_name]['target_col']
    y_test_original = dataset[test_mask][target_col]
    
    # Remove test samples with NaN
    test_clean_mask = ~(X_test_original.isnull().any(axis=1) | y_test_original.isnull())
    X_test_clean = X_test_original[test_clean_mask]
    y_test_clean = y_test_original[test_clean_mask]
    
    print(f"\n{'='*50}")
    print(f"TRAINING: {target_name}")
    print(f"{'='*50}")
    
    for strategy_name, strategy_data in strategies.items():
        if 'error' in strategy_data:
            continue
            
        print(f"\nStrategy: {strategy_name}")
        print(f"Training samples: {strategy_data['size']:,}")
        
        X_train_aug = strategy_data['X']
        y_train_aug = strategy_data['y']
        
        # Encode labels for sklearn
        le = LabelEncoder()
        le.fit(pd.concat([pd.Series(y_train_aug), y_test_clean], axis=0).astype(str))
        y_train_encoded = le.transform(pd.Series(y_train_aug).astype(str))
        y_test_encoded = le.transform(y_test_clean.astype(str))
        
        # Train Random Forest (best performing model from previous analysis)
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        
        print("  Training Random Forest...")
        rf_model.fit(X_train_aug, y_train_encoded)
        
        print("  Making predictions...")
        y_pred_encoded = rf_model.predict(X_test_clean)
        y_pred_decoded = le.inverse_transform(y_pred_encoded)
        
        # Calculate metrics
        report = classification_report(y_test_clean.astype(str), y_pred_decoded.astype(str), 
                                     digits=4, output_dict=True)
        
        accuracy = report['accuracy']
        macro_f1 = report['macro avg']['f1-score']
        weighted_f1 = report['weighted avg']['f1-score']
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Macro F1: {macro_f1:.4f}")
        print(f"  Weighted F1: {weighted_f1:.4f}")
        
        # Store results
        augmented_results.append({
            'target_name': target_name,
            'strategy': strategy_name,
            'training_size': strategy_data['size'],
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'class_distribution': dict(strategy_data['distribution'])
        })

# Create comparison DataFrame
augmented_df = pd.DataFrame(augmented_results)

print(f"\n{'='*80}")
print("AUGMENTATION RESULTS COMPARISON")
print(f"{'='*80}")

if not augmented_df.empty:
    # Display results grouped by target
    for target in augmented_df['target_name'].unique():
        target_results = augmented_df[augmented_df['target_name'] == target].copy()
        target_results = target_results.sort_values('accuracy', ascending=False)
        
        print(f"\n{target}:")
        print(target_results[['strategy', 'training_size', 'accuracy', 'macro_f1', 'weighted_f1']].to_string(index=False))
        
        # Calculate improvement over original
        original_acc = target_results[target_results['strategy'] == 'original']['accuracy'].values
        if len(original_acc) > 0:
            original_acc = original_acc[0]
            print(f"\nImprovements over original ({original_acc:.4f}):")
            for _, row in target_results.iterrows():
                if row['strategy'] != 'original':
                    improvement = row['accuracy'] - original_acc
                    improvement_pct = (improvement / original_acc) * 100
                    print(f"  {row['strategy']}: {improvement:+.4f} ({improvement_pct:+.2f}%)")

    # Save augmented results
    if 'results_folder' in locals():
        augmented_df.to_csv(os.path.join(results_folder, 'augmentation_results.csv'), index=False)
        print(f"\nAugmentation results saved to {results_folder}/augmentation_results.csv")
else:
    print("No augmented results to display")

In [ ]:
# Visualize augmentation effects and provide recommendations
if not augmented_df.empty:
    print("=== Visualizing Augmentation Effects ===")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Data Augmentation Impact Analysis', fontsize=16, fontweight='bold')
    
    targets = augmented_df['target_name'].unique()
    
    for i, target in enumerate(targets):
        target_data = augmented_df[augmented_df['target_name'] == target].copy()
        
        # Plot 1: Accuracy vs Training Size
        ax1 = axes[i, 0]
        colors = ['red' if x == 'original' else 'blue' for x in target_data['strategy']]
        scatter = ax1.scatter(target_data['training_size'], target_data['accuracy'], 
                            c=colors, s=100, alpha=0.7)
        
        # Add strategy labels
        for _, row in target_data.iterrows():
            ax1.annotate(row['strategy'], (row['training_size'], row['accuracy']),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax1.set_xlabel('Training Set Size')
        ax1.set_ylabel('Accuracy')
        ax1.set_title(f'{target}\nAccuracy vs Dataset Size')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Metric Comparison
        ax2 = axes[i, 1]
        metrics = ['accuracy', 'macro_f1', 'weighted_f1']
        x_pos = np.arange(len(target_data))
        
        width = 0.25
        for j, metric in enumerate(metrics):
            ax2.bar(x_pos + j*width, target_data[metric], width, 
                   label=metric.replace('_', ' ').title(), alpha=0.8)
        
        ax2.set_xlabel('Augmentation Strategy')
        ax2.set_ylabel('Score')
        ax2.set_title(f'{target}\nMetric Comparison')
        ax2.set_xticks(x_pos + width)
        ax2.set_xticklabels(target_data['strategy'], rotation=45, ha='right')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    if 'results_folder' in locals():
        plot_path = os.path.join(results_folder, 'augmentation_analysis.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f"Augmentation analysis plot saved to: {plot_path}")
    
    plt.show()
    
    # Final recommendations
    print(f"\n{'='*80}")
    print("FINAL AUGMENTATION RECOMMENDATIONS")
    print(f"{'='*80}")
    
    for target in targets:
        target_data = augmented_df[augmented_df['target_name'] == target].copy()
        target_data = target_data.sort_values('accuracy', ascending=False)
        
        original_row = target_data[target_data['strategy'] == 'original']
        best_augmented = target_data[target_data['strategy'] != 'original'].iloc[0] if len(target_data) > 1 else None
        
        print(f"\n{target}:")
        if original_row.empty:
            print("  No original baseline found")
            continue
            
        original_acc = original_row['accuracy'].iloc[0]
        original_size = original_row['training_size'].iloc[0]
        
        print(f"  Original performance: {original_acc:.4f} (n={original_size:,})")
        
        if best_augmented is not None:
            best_acc = best_augmented['accuracy']
            best_size = best_augmented['training_size']
            best_strategy = best_augmented['strategy']
            
            improvement = best_acc - original_acc
            improvement_pct = (improvement / original_acc) * 100
            size_increase = ((best_size - original_size) / original_size) * 100
            
            print(f"  Best augmented: {best_acc:.4f} (n={best_size:,}) using {best_strategy}")
            print(f"  Improvement: {improvement:+.4f} ({improvement_pct:+.2f}%)")
            print(f"  Dataset size increase: +{size_increase:.1f}%")
            
            if improvement > 0.01:  # 1% improvement threshold
                print(f"  🎯 RECOMMENDATION: Use {best_strategy} - significant improvement!")
            elif improvement > 0.005:  # 0.5% improvement threshold
                print(f"  ✅ RECOMMENDATION: Use {best_strategy} - moderate improvement")
            else:
                print(f"  ⚠️  RECOMMENDATION: Original dataset sufficient - minimal improvement")
        else:
            print("  No successful augmentation strategies")
    
    # Overall summary
    overall_improvements = []
    for target in targets:
        target_data = augmented_df[augmented_df['target_name'] == target]
        original_acc = target_data[target_data['strategy'] == 'original']['accuracy'].values
        if len(original_acc) > 0:
            best_aug_acc = target_data[target_data['strategy'] != 'original']['accuracy'].max()
            if not np.isnan(best_aug_acc):
                improvement = best_aug_acc - original_acc[0]
                overall_improvements.append(improvement)
    
    if overall_improvements:
        avg_improvement = np.mean(overall_improvements)
        print(f"\n📊 OVERALL IMPACT:")
        print(f"   Average accuracy improvement: {avg_improvement:+.4f}")
        print(f"   Targets with improvement: {sum(1 for x in overall_improvements if x > 0)}/{len(overall_improvements)}")
        
        if avg_improvement > 0.01:
            print(f"   🚀 Data augmentation provides significant benefits!")
        elif avg_improvement > 0:
            print(f"   ✨ Data augmentation provides modest benefits")
        else:
            print(f"   📝 Original datasets are sufficient for this task")
            
else:
    print("No augmentation results available for visualization")

# Dimensionality Reduction & Feature Separation Analysis

This section uses t-SNE, UMAP, and PCA to visualize how network traffic features separate different QoE classes.
We'll explore whether the features naturally cluster by QoE labels, which indicates model predictability.

# Feature Importance Analysis

This section analyzes which network traffic features are most important for predicting QoE.
We'll show feature importance for the best performing models only.

In [ ]:
# Analyze feature importance for the best models only
print("=== Feature Importance Analysis ===")

if 'trained_model' in locals() and trained_model:
    # Find the best models for each target/variant combination
    best_models_for_importance = {}
    
    # Group by target and variant to find best model for each
    for tm in trained_model:
        key = (tm['target_name'], tm['variant'])
        accuracy = tm['report']['accuracy']
        
        if key not in best_models_for_importance or accuracy > best_models_for_importance[key]['accuracy']:
            best_models_for_importance[key] = {
                'model_info': tm,
                'accuracy': accuracy
            }
    
    print(f"Found {len(best_models_for_importance)} best model combinations")
    
    # Analyze feature importance for models that support it
    for (target_name, variant), best_info in best_models_for_importance.items():
        model_info = best_info['model_info']
        model = model_info['model']
        accuracy = best_info['accuracy']
        
        # Only analyze multiclass models (as requested)
        if variant != 'multiclass':
            continue
            
        print(f"\n{'='*60}")
        print(f"FEATURE IMPORTANCE: {target_name} ({variant})")
        print(f"Model: {model_info['model_name']} | Accuracy: {accuracy:.4f}")
        print(f"{'='*60}")
        
        # Check if model has feature importance
        if hasattr(model, 'feature_importances_'):
            # Create feature importance DataFrame
            importance_df = pd.DataFrame({
                'feature': features,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)
            
            print(f"\nTop 15 Most Important Features:")
            print("-" * 50)
            for i, (_, row) in enumerate(importance_df.head(15).iterrows(), 1):
                print(f"{i:2d}. {row['feature']:<35} {row['importance']:.4f}")
            
            # Calculate cumulative importance
            importance_df['cumulative'] = importance_df['importance'].cumsum()
            
            # Find features contributing to 80% of importance
            features_80_pct = len(importance_df[importance_df['cumulative'] <= 0.8])
            print(f"\nFeatures contributing to 80% of importance: {features_80_pct}/{len(features)}")
            
            # Create visualization
            plt.figure(figsize=(15, 10))
            
            # Plot 1: Top 15 features bar plot
            plt.subplot(2, 2, 1)
            top_features = importance_df.head(15)
            bars = plt.barh(range(len(top_features)), top_features['importance'])
            plt.yticks(range(len(top_features)), top_features['feature'])
            plt.xlabel('Feature Importance')
            plt.title(f'Top 15 Features - {target_name}\\n{model_info["model_name"]} (Acc: {accuracy:.3f})')
            plt.gca().invert_yaxis()
            
            # Color bars by importance level
            colors = ['darkgreen' if x > 0.05 else 'green' if x > 0.03 else 'orange' if x > 0.02 else 'lightcoral' for x in top_features['importance']]
            for bar, color in zip(bars, colors):
                bar.set_color(color)
            
            # Plot 2: Cumulative importance
            plt.subplot(2, 2, 2)
            plt.plot(range(1, len(importance_df) + 1), importance_df['cumulative'], 'b-', linewidth=2)
            plt.axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='80% threshold')
            plt.axhline(y=0.9, color='orange', linestyle='--', alpha=0.7, label='90% threshold')
            plt.xlabel('Number of Features')
            plt.ylabel('Cumulative Importance')
            plt.title('Cumulative Feature Importance')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            # Plot 3: Feature categories analysis
            plt.subplot(2, 2, 3)
            # Group features by category
            categories = {
                'src2dst_pkt_size': [],
                'src2dst_pps': [],
                'dst2src_pkt_size': [],
                'dst2src_pps': [],
                'pkt_dir_ratio': []
            }
            
            for _, row in importance_df.iterrows():
                feature = row['feature']
                importance = row['importance']
                if 'src2dst' in feature and 'pkt_size' in feature:
                    categories['src2dst_pkt_size'].append(importance)
                elif 'src2dst' in feature and 'pps' in feature:
                    categories['src2dst_pps'].append(importance)
                elif 'dst2src' in feature and 'pkt_size' in feature:
                    categories['dst2src_pkt_size'].append(importance)
                elif 'dst2src' in feature and 'pps' in feature:
                    categories['dst2src_pps'].append(importance)
                elif 'pkt_dir_ratio' in feature:
                    categories['pkt_dir_ratio'].append(importance)
            
            # Calculate total importance per category
            category_totals = {cat: sum(values) for cat, values in categories.items()}
            
            labels = list(category_totals.keys())
            values = list(category_totals.values())
            
            plt.pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
            plt.title('Importance by Feature Category')
            
            # Plot 4: Distribution of importance values
            plt.subplot(2, 2, 4)
            plt.hist(importance_df['importance'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            plt.xlabel('Feature Importance')
            plt.ylabel('Number of Features')
            plt.title('Distribution of Feature Importance')
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            
            # Save plot
            if 'results_folder' in locals():
                plot_path = os.path.join(results_folder, f'feature_importance_{target_name.replace(" ", "_")}_{variant}.png')
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                print(f"\\nFeature importance plot saved to: {plot_path}")
            
            plt.show()
            
            # Save feature importance data
            if 'results_folder' in locals():
                csv_path = os.path.join(results_folder, f'feature_importance_{target_name.replace(" ", "_")}_{variant}.csv')
                importance_df.to_csv(csv_path, index=False)
                print(f"Feature importance data saved to: {csv_path}")
            
            # Insights
            print(f"\\n💡 Key Insights for {target_name}:")
            top_3 = importance_df.head(3)
            print(f"   • Most important: {top_3.iloc[0]['feature']} ({top_3.iloc[0]['importance']:.4f})")
            print(f"   • Second most: {top_3.iloc[1]['feature']} ({top_3.iloc[1]['importance']:.4f})")
            print(f"   • Third most: {top_3.iloc[2]['feature']} ({top_3.iloc[2]['importance']:.4f})")
            
            # Category insights
            top_category = max(category_totals, key=category_totals.get)
            print(f"   • Most important category: {top_category} ({category_totals[top_category]:.4f} total)")
            
        else:
            print(f"Model {model_info['model_name']} does not support feature importance analysis")
            print(f"Available methods: {[method for method in dir(model) if 'importance' in method.lower()]}")

else:
    print("⚠️ No trained models available for feature importance analysis")
    print("Please run the model training section first")

In [ ]:
# Install and import libraries for dimensionality reduction
import subprocess
import sys

# Check for required libraries
required_libs = {
    'umap-learn': 'umap',
    'plotly': 'plotly',
    'nbformat': 'nbformat'
}

for lib_name, import_name in required_libs.items():
    try:
        if import_name == 'nbformat':
            import nbformat
            # Check version requirement
            try:
                from packaging import version
                if version.parse(nbformat.__version__) < version.parse("4.2.0"):
                    raise ImportError("nbformat version too old")
            except ImportError:
                # If packaging is not available, install it and try again
                subprocess.check_call([sys.executable, "-m", "pip", "install", "packaging"])
                from packaging import version
                if version.parse(nbformat.__version__) < version.parse("4.2.0"):
                    raise ImportError("nbformat version too old")
        else:
            __import__(import_name)
        print(f"✓ {lib_name} already installed")
    except ImportError:
        print(f"Installing {lib_name}...")
        if lib_name == 'nbformat':
            subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat>=4.2.0"])
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib_name])
        print(f"✓ {lib_name} installed successfully")

# Import libraries
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
try:
    # Try the correct import for umap-learn
    from umap.umap_ import UMAP
    umap_available = True
    print("✓ UMAP imported successfully")
except ImportError:
    try:
        # Fallback for older versions
        from umap import UMAP
        umap_available = True
        print("✓ UMAP imported successfully (fallback)")
    except ImportError:
        print("⚠ UMAP not available - will skip UMAP analysis")
        umap_available = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from sklearn.metrics import silhouette_score, calinski_harabasz_score

print("✓ All dimensionality reduction libraries ready")

In [ ]:
# Prepare data for dimensionality reduction analysis
print("=== Preparing Data for Dimensionality Reduction ===")

# Create a sample of the data for faster computation (t-SNE can be slow on large datasets)
def create_analysis_sample(df, features, max_samples=5000, random_state=42):
    """Create a stratified sample for analysis"""
    if len(df) <= max_samples:
        return df
    
    # Try to maintain class proportions in sample
    sample_df = df.sample(n=max_samples, random_state=random_state)
    return sample_df

# Prepare datasets for each target
dr_datasets = {}

for target_name, target_col in target_list:
    print(f"\n{target_name}:")
    
    # Get clean data (training + test for visualization)
    all_data = dataset[features + [target_col]].dropna()
    
    print(f"  Total clean samples: {len(all_data):,}")
    
    # Create sample for analysis
    sample_data = create_analysis_sample(all_data, features, max_samples=5000)
    print(f"  Analysis sample: {len(sample_data):,}")
    
    # Prepare features and labels
    X_sample = sample_data[features]
    y_sample = sample_data[target_col]
    
    # Standardize features (important for t-SNE and UMAP)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sample)
    
    # Store for analysis
    dr_datasets[target_name] = {
        'X': X_sample,
        'X_scaled': X_scaled,
        'y': y_sample,
        'scaler': scaler,
        'n_samples': len(sample_data),
        'class_counts': dict(y_sample.value_counts()),
        'feature_names': features
    }
    
    print(f"  Class distribution: {dict(y_sample.value_counts())}")
    print(f"  Feature matrix shape: {X_scaled.shape}")

print(f"\nReady for dimensionality reduction analysis on {len(dr_datasets)} targets")

In [ ]:
# Apply dimensionality reduction techniques
print("=== Applying Dimensionality Reduction Techniques ===")

# Define reduction methods
reduction_methods = {
    'PCA': lambda X: PCA(n_components=2, random_state=42).fit_transform(X),
    't-SNE': lambda X: TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000).fit_transform(X)
}

# Add UMAP if available
if umap_available:
    reduction_methods['UMAP'] = lambda X: UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1).fit_transform(X)
    print("✓ UMAP method added")
else:
    print("⚠ UMAP method skipped (not available)")

# Store results
dr_results = {}

for target_name, data in dr_datasets.items():
    print(f"\n{'='*50}")
    print(f"TARGET: {target_name}")
    print(f"{'='*50}")
    
    X_scaled = data['X_scaled']
    y = data['y']
    
    dr_results[target_name] = {
        'original_data': data,
        'embeddings': {},
        'separation_metrics': {}
    }
    
    # Apply each reduction method
    for method_name, method_func in reduction_methods.items():
        print(f"\nApplying {method_name}...")
        
        try:
            # Apply dimensionality reduction
            embedding = method_func(X_scaled)
            
            # Store embedding
            dr_results[target_name]['embeddings'][method_name] = embedding
            
            # Calculate separation metrics
            print(f"  Computing separation metrics...")
            
            # Silhouette score (higher = better separation)
            sil_score = silhouette_score(embedding, y)
            
            # Calinski-Harabasz score (higher = better separation)
            ch_score = calinski_harabasz_score(embedding, y)
            
            # Custom separation metric: average distance between class centroids
            class_centroids = {}
            for class_val in np.unique(y):
                class_mask = (y == class_val)
                class_centroids[class_val] = np.mean(embedding[class_mask], axis=0)
            
            # Calculate pairwise distances between centroids
            centroid_distances = []
            class_vals = list(class_centroids.keys())
            for i in range(len(class_vals)):
                for j in range(i+1, len(class_vals)):
                    dist = np.linalg.norm(class_centroids[class_vals[i]] - class_centroids[class_vals[j]])
                    centroid_distances.append(dist)
            
            avg_centroid_distance = np.mean(centroid_distances) if centroid_distances else 0
            
            # Store metrics
            dr_results[target_name]['separation_metrics'][method_name] = {
                'silhouette_score': sil_score,
                'calinski_harabasz_score': ch_score,
                'avg_centroid_distance': avg_centroid_distance,
                'embedding_range': [embedding.min(), embedding.max()],
                'embedding_std': np.std(embedding)
            }
            
            print(f"  ✓ {method_name} completed")
            print(f"    Silhouette Score: {sil_score:.3f}")
            print(f"    Calinski-Harabasz Score: {ch_score:.1f}")
            print(f"    Avg Centroid Distance: {avg_centroid_distance:.3f}")
            
        except Exception as e:
            print(f"  ✗ {method_name} failed: {str(e)}")
            dr_results[target_name]['separation_metrics'][method_name] = {'error': str(e)}

# Summary of separation quality
print(f"\n{'='*80}")
print("SEPARATION QUALITY SUMMARY")
print(f"{'='*80}")

separation_summary = []
for target_name, results in dr_results.items():
    print(f"\n{target_name}:")
    
    for method_name, metrics in results['separation_metrics'].items():
        if 'error' not in metrics:
            sil = metrics['silhouette_score']
            ch = metrics['calinski_harabasz_score']
            centroid_dist = metrics['avg_centroid_distance']
            
            # Overall separation quality score (normalized combination of metrics)
            # Silhouette: -1 to 1 (higher better), normalize to 0-1
            sil_norm = (sil + 1) / 2
            # CH score: normalize by taking log and scaling
            ch_norm = min(1.0, np.log(ch + 1) / 10)
            # Centroid distance: normalize by embedding std
            centroid_norm = min(1.0, centroid_dist / (metrics['embedding_std'] + 1e-6))
            
            # Combined score
            combined_score = (sil_norm + ch_norm + centroid_norm) / 3
            
            print(f"  {method_name:8s}: Silhouette={sil:.3f}, CH={ch:.1f}, Combined={combined_score:.3f}")
            
            separation_summary.append({
                'target': target_name,
                'method': method_name,
                'silhouette': sil,
                'calinski_harabasz': ch,
                'centroid_distance': centroid_dist,
                'combined_score': combined_score
            })
        else:
            print(f"  {method_name:8s}: Failed - {metrics['error']}")

# Convert to DataFrame for easy analysis
separation_df = pd.DataFrame(separation_summary)

if not separation_df.empty:
    print(f"\n=== Best Separation Methods ===")
    # Find best method for each target
    for target in separation_df['target'].unique():
        target_data = separation_df[separation_df['target'] == target]
        best_method = target_data.loc[target_data['combined_score'].idxmax()]
        print(f"{target}: {best_method['method']} (score: {best_method['combined_score']:.3f})")
    
    # Overall best methods
    method_avg_scores = separation_df.groupby('method')['combined_score'].mean().sort_values(ascending=False)
    print(f"\nOverall method ranking:")
    for method, score in method_avg_scores.items():
        print(f"  {method}: {score:.3f}")
        
else:
    print("No separation metrics available")

In [ ]:
# Create comprehensive visualizations of dimensionality reduction results
print("=== Creating Dimensionality Reduction Visualizations ===")

if dr_results:
    # Create matplotlib visualizations
    for target_name, results in dr_results.items():
        if not results['embeddings']:
            continue
            
        print(f"\nCreating plots for {target_name}...")
        
        # Set up the plot
        n_methods = len(results['embeddings'])
        fig, axes = plt.subplots(1, n_methods, figsize=(6*n_methods, 5))
        if n_methods == 1:
            axes = [axes]
            
        fig.suptitle(f'{target_name} - Feature Separation Analysis', fontsize=16, fontweight='bold')
        
        y = results['original_data']['y']
        unique_classes = sorted(y.unique())
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique_classes)))
        
        for i, (method_name, embedding) in enumerate(results['embeddings'].items()):
            ax = axes[i]
            
            # Plot each class with different colors
            for j, class_val in enumerate(unique_classes):
                mask = (y == class_val)
                ax.scatter(embedding[mask, 0], embedding[mask, 1], 
                          c=[colors[j]], label=f'Class {class_val}', 
                          alpha=0.6, s=20)
            
            # Get separation metrics for title
            if method_name in results['separation_metrics']:
                metrics = results['separation_metrics'][method_name]
                if 'error' not in metrics:
                    sil_score = metrics['silhouette_score']
                    title = f'{method_name}\nSilhouette: {sil_score:.3f}'
                else:
                    title = f'{method_name}\n(Failed)'
            else:
                title = method_name
                
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel(f'{method_name} Component 1')
            ax.set_ylabel(f'{method_name} Component 2')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save plot
        if 'results_folder' in locals():
            plot_path = os.path.join(results_folder, f'dimensionality_reduction_{target_name.replace(" ", "_")}.png')
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            print(f"  Saved: {plot_path}")
        
        plt.show()

    # Create interactive Plotly visualizations for the best method
    print(f"\nCreating interactive visualizations...")
    
    # Find the overall best method
    if not separation_df.empty:
        overall_best_method = separation_df.groupby('method')['combined_score'].mean().idxmax()
        print(f"Using {overall_best_method} for interactive plots (best overall method)")
        
        # Create interactive plots for each target
        for target_name, results in dr_results.items():
            if overall_best_method not in results['embeddings']:
                continue
                
            embedding = results['embeddings'][overall_best_method]
            y = results['original_data']['y']
            
            # Create DataFrame for Plotly
            plot_df = pd.DataFrame({
                'Component_1': embedding[:, 0],
                'Component_2': embedding[:, 1],
                'QoE_Class': y.astype(str),
                'Sample_Index': range(len(y))
            })
            
            # Create interactive scatter plot
            fig = px.scatter(
                plot_df, 
                x='Component_1', 
                y='Component_2', 
                color='QoE_Class',
                title=f'{target_name} - {overall_best_method} Visualization',
                labels={
                    'Component_1': f'{overall_best_method} Component 1',
                    'Component_2': f'{overall_best_method} Component 2',
                    'QoE_Class': 'QoE Class'
                },
                hover_data=['Sample_Index']
            )
            
            # Update layout
            fig.update_layout(
                width=800,
                height=600,
                title_font_size=16,
                legend_title_font_size=14
            )
            
            # Save interactive plot
            if 'results_folder' in locals():
                interactive_path = os.path.join(results_folder, f'interactive_{overall_best_method}_{target_name.replace(" ", "_")}.html')
                fig.write_html(interactive_path)
                print(f"  Saved interactive plot: {interactive_path}")
            
            # Try to show interactive plot, fallback to static if there are display issues
            try:
                fig.show()
            except Exception as e:
                print(f"  ⚠️ Interactive plot display failed: {e}")
                print(f"  📁 Interactive plot saved to file: {interactive_path if 'interactive_path' in locals() else 'N/A'}")
                # Create a static matplotlib fallback
                plt.figure(figsize=(10, 8))
                
                # Plot each class with different colors
                unique_classes = sorted(plot_df['QoE_Class'].unique())
                colors = plt.cm.Set3(np.linspace(0, 1, len(unique_classes)))
                
                for i, class_val in enumerate(unique_classes):
                    mask = plot_df['QoE_Class'] == class_val
                    plt.scatter(plot_df[mask]['Component_1'], plot_df[mask]['Component_2'], 
                               c=[colors[i]], label=f'Class {class_val}', alpha=0.6, s=20)
                
                plt.xlabel(f'{overall_best_method} Component 1')
                plt.ylabel(f'{overall_best_method} Component 2')
                plt.title(f'{target_name} - {overall_best_method} Visualization (Static Fallback)')
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

    # Create separation metrics comparison plot
    if not separation_df.empty:
        print(f"\nCreating separation metrics comparison...")
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle('Dimensionality Reduction Method Comparison', fontsize=16, fontweight='bold')
        
        metrics = ['silhouette', 'calinski_harabasz', 'combined_score']
        metric_titles = ['Silhouette Score', 'Calinski-Harabasz Score', 'Combined Separation Score']
        
        for i, (metric, title) in enumerate(zip(metrics, metric_titles)):
            pivot_data = separation_df.pivot(index='target', columns='method', values=metric)
            
            # Create heatmap
            sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlBu_r', 
                       ax=axes[i], cbar_kws={'label': title})
            axes[i].set_title(title, fontweight='bold')
            axes[i].set_xlabel('Method')
            axes[i].set_ylabel('Target Type')
        
        plt.tight_layout()
        
        # Save comparison plot
        if 'results_folder' in locals():
            comparison_path = os.path.join(results_folder, 'method_comparison_heatmap.png')
            plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
            print(f"  Saved comparison: {comparison_path}")
        
        plt.show()

else:
    print("No dimensionality reduction results available for visualization")

In [ ]:
# Interpret dimensionality reduction results and provide insights
print("=== Dimensionality Reduction Analysis & Insights ===")

if not separation_df.empty:
    # Save detailed results
    if 'results_folder' in locals():
        separation_df.to_csv(os.path.join(results_folder, 'dimensionality_reduction_metrics.csv'), index=False)
        print(f"Detailed metrics saved to {results_folder}/dimensionality_reduction_metrics.csv")
    
    # Detailed analysis for each target
    for target_name in separation_df['target'].unique():
        target_data = separation_df[separation_df['target'] == target_name]
        target_results = dr_results[target_name]
        
        print(f"\n{'='*60}")
        print(f"ANALYSIS: {target_name}")
        print(f"{'='*60}")
        
        # Get class distribution
        class_counts = target_results['original_data']['class_counts']
        n_samples = target_results['original_data']['n_samples']
        n_features = len(target_results['original_data']['feature_names'])
        
        print(f"Dataset characteristics:")
        print(f"  • Samples: {n_samples:,}")
        print(f"  • Features: {n_features}")
        print(f"  • Classes: {len(class_counts)}")
        print(f"  • Class distribution: {class_counts}")
        
        # Calculate class balance
        class_values = list(class_counts.values())
        balance_ratio = max(class_values) / min(class_values) if min(class_values) > 0 else float('inf')
        print(f"  • Class imbalance ratio: {balance_ratio:.2f}:1")
        
        # Method performance analysis
        print(f"\nDimensionality reduction performance:")
        target_sorted = target_data.sort_values('combined_score', ascending=False)
        
        for _, row in target_sorted.iterrows():
            method = row['method']
            sil = row['silhouette']
            combined = row['combined_score']
            
            # Interpret silhouette score
            if sil > 0.5:
                sil_interp = "Excellent separation"
            elif sil > 0.3:
                sil_interp = "Good separation"
            elif sil > 0.1:
                sil_interp = "Moderate separation"
            elif sil > 0:
                sil_interp = "Weak separation"
            else:
                sil_interp = "Poor/overlapping clusters"
            
            print(f"  • {method:8s}: Combined={combined:.3f}, Silhouette={sil:.3f} ({sil_interp})")
        
        # Best method recommendation
        best_method_row = target_sorted.iloc[0]
        best_method = best_method_row['method']
        best_score = best_method_row['combined_score']
        
        print(f"\n🎯 Recommendation for {target_name}:")
        print(f"   Best method: {best_method} (score: {best_score:.3f})")
        
        if best_score > 0.6:
            print(f"   ✅ Excellent feature separation - QoE classes are well-distinguishable")
            print(f"   ✅ Machine learning models should perform very well on this target")
        elif best_score > 0.4:
            print(f"   ✨ Good feature separation - QoE classes have distinct patterns")
            print(f"   ✨ Machine learning models should perform well with proper tuning")
        elif best_score > 0.25:
            print(f"   ⚠️  Moderate separation - some overlap between QoE classes")
            print(f"   ⚠️  Consider feature engineering or ensemble methods")
        else:
            print(f"   ❌ Poor separation - QoE classes heavily overlap in feature space")
            print(f"   ❌ This target may be difficult to predict accurately")
        
        # Feature space insights
        if best_method in target_results['separation_metrics']:
            metrics = target_results['separation_metrics'][best_method]
            if 'error' not in metrics:
                centroid_dist = metrics['avg_centroid_distance']
                embedding_std = metrics['embedding_std']
                
                print(f"\n📊 Feature space insights:")
                print(f"   • Class centroids are {centroid_dist:.3f} units apart on average")
                print(f"   • Data spread (std): {embedding_std:.3f}")
                
                # Relative separation
                if embedding_std > 0:
                    relative_sep = centroid_dist / embedding_std
                    if relative_sep > 2:
                        print(f"   • Classes are well-separated relative to data spread")
                    elif relative_sep > 1:
                        print(f"   • Classes have moderate separation")
                    else:
                        print(f"   • Classes overlap significantly")

    # Overall conclusions
    print(f"\n{'='*80}")
    print("OVERALL CONCLUSIONS")
    print(f"{'='*80}")
    
    # Method comparison
    method_avg = separation_df.groupby('method').agg({
        'silhouette': 'mean',
        'combined_score': 'mean'
    }).round(3)
    
    print(f"\nMethod performance across all targets:")
    for method, scores in method_avg.iterrows():
        sil_avg = scores['silhouette']
        combined_avg = scores['combined_score']
        print(f"  {method:8s}: Avg Silhouette={sil_avg:.3f}, Avg Combined={combined_avg:.3f}")
    
    # Best overall method
    best_overall = method_avg['combined_score'].idxmax()
    best_score = method_avg.loc[best_overall, 'combined_score']
    
    print(f"\n🏆 Best overall method: {best_overall} (avg score: {best_score:.3f})")
    
    # Predictability assessment
    avg_scores = separation_df.groupby('target')['combined_score'].mean()
    
    print(f"\nPredictability assessment:")
    for target, score in avg_scores.items():
        if score > 0.5:
            assessment = "Highly predictable"
        elif score > 0.35:
            assessment = "Moderately predictable"
        else:
            assessment = "Challenging to predict"
        
        print(f"  {target}: {score:.3f} - {assessment}")
    
    # Feature engineering recommendations
    print(f"\n💡 Feature Engineering Recommendations:")
    low_sep_targets = avg_scores[avg_scores < 0.4].index.tolist()
    
    if low_sep_targets:
        print(f"   For targets with low separation {low_sep_targets}:")
        print(f"   • Consider adding temporal features (rate of change, trends)")
        print(f"   • Try feature interactions (ratios, products)")
        print(f"   • Apply non-linear transformations (log, sqrt)")
        print(f"   • Use domain-specific gaming metrics")
    else:
        print(f"   ✅ Current features provide good separation for all targets")
        print(f"   ✅ Focus on model optimization rather than feature engineering")
    
    # Data augmentation insights
    high_sep_targets = avg_scores[avg_scores > 0.5].index.tolist()
    if high_sep_targets:
        print(f"\n🔬 Data Augmentation Insights:")
        print(f"   Targets with good separation {high_sep_targets}:")
        print(f"   • SMOTE should work very well (clear class boundaries)")
        print(f"   • Synthetic samples will likely be realistic")
        print(f"   • Consider advanced SMOTE variants (BorderlineSMOTE)")
    
else:
    print("No separation analysis results available")

print(f"\n✅ Dimensionality reduction analysis complete!")